### Global Configuration

This cell defines the overall experiment setup for the tutorial. In most cases, this is the main place where you may need to make changes before creating the FABRIC slice.

**What this cell does:**
- Sets the slice name and node image
- Chooses which FABRIC site to use for each node
- Defines the names of nodes, networks, and NICs
- Specifies the resource size of each node, including cores, RAM, and disk

**Things you may want to change:**
- `SLICE_NAME`: the name of your FABRIC slice
- `SITES`: the FABRIC site used for `UERAN`, `CN`, and `DN`
- `PROFILE`: the resource size of each node

**Notes:**
- For this tutorial, all nodes are typically placed at the same FABRIC site.
- Make sure the site you choose has enough available resources.
- The CN node is configured with more resources because it runs the main L25GC+ components.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as FablibManager
import uuid
import time

# ================
# Global config
# ================
# Notes:
# - Keep slice name, image, site mapping, node/network/NIC naming, and resource profiles here.
# - In most cases, you only need to edit this cell to change the sites and resource sizing.

SLICE_NAME = f"L25GC-{uuid.uuid4().hex[:8]}"
NODE_IMAGE = "default_ubuntu_22"

# Sites: which FABRIC site to place each role
SITES = {
    "UERAN": "TACC",
    "CN":    "TACC",
    "DN":    "TACC",
}

# Node names: what you will see in FABRIC portal / fablib
NODE = {
    "UERAN": "ueran_node",
    "CN":    "cn_node",
    "DN":    "dn_node",
}

# Network names
NET = {
    "UERAN_CN_CP": "subnet-1",  # control plane (N2)
    "UERAN_CN_DP": "subnet-2",  # data plane (N3)
    "CN_DN":       "subnet-3",  # data plane (N6)
}

# NIC component names: each NIC_Basic component name
NIC = {
    "UERAN_CP":        "n2",  # N2 interface on UERAN node
    "UERAN_DP":        "n3",  # N3 interface on UERAN node
    "CN_CP":           "n2",  # N2 interface on CN node
    "CN_DP_TO_UERAN":  "n3",  # N3 interface on CN node
    "CN_DP_TO_DN":     "n6",  # N6 interface on CN node
    "DN_DP":           "n6",  # N6 interface on DN node
}

# Node resource profiles
PROFILE = {
    "UERAN": dict(cores=5, ram=16, disk=10),
    "CN":    dict(cores=20, ram=64, disk=40),
    "DN":    dict(cores=5, ram=16, disk=10),
}


### Initialize FABRIC Client

This cell initializes the FABRIC Python client and tells the notebook which FABRIC project to use.

**What this cell does:**
- Creates the `fablib` object used throughout the notebook
- Associates the session with a specific FABRIC project

**Things you may want to change:**
- `project_id`, if you want to use a different FABRIC project

**Notes:**
- Your FABRIC credentials must already be configured correctly.
- Your account must have access to the selected FABRIC project.

In [ ]:
# ================
# Initialize Fablib
# ================

# Notes:
# - FablibManager is the entry point for interacting with FABRIC.
# - If you want to inspect your fablib configuration, uncomment fablib.show_config().

# Tutorial project ID
project_id = 'a7818636-1fa1-4e77-bb03-d171598b0862'

# fablib = FablibManager()
fablib = FablibManager(project_id=project_id)

# fablib.show_config();

### Helper Functions

This cell defines helper functions used by the rest of the notebook. These functions simplify common tasks such as creating nodes, attaching NICs, configuring interfaces, and printing network information.

**What this cell does:**
- Defines reusable helper functions for topology creation and network setup
- Keeps later cells cleaner and easier to read

**Notes:**
- In normal usage, you do not need to modify this cell.
- If you change this cell, make sure you understand how later cells depend on these helper functions.

In [ ]:
# ================
# Helper functions
# ================

# Notes:
# - These helpers encapsulate repetitive operations: creating nodes, NICs, networks, assigning IPs, etc.
# - Keeping this logic in functions makes the workflow cells much cleaner.

def add_basic_nic(node, component_name: str):
    """Add a NIC_Basic component and return its first interface."""
    iface = node.add_component(model="NIC_Basic", name=component_name).get_interfaces()[0]
    iface.set_mode('auto')

    return iface


def add_node(slice_obj, role: str):
    """
    Create a node for a given role and attach its NICs.

    Returns:
      node: fablib node object
      ifaces: dict[str, interface] keyed by semantic role name
    """
    node = slice_obj.add_node(
        name=NODE[role],
        site=SITES[role],
        image=NODE_IMAGE,
        **PROFILE[role],
    )

    if role == "UERAN":
        ifaces = {
            "CP": add_basic_nic(node, NIC["UERAN_CP"]),
            "DP": add_basic_nic(node, NIC["UERAN_DP"]),
        }
    elif role == "CN":
        ifaces = {
            "CP":          add_basic_nic(node, NIC["CN_CP"]),
            "DP_TO_UERAN": add_basic_nic(node, NIC["CN_DP_TO_UERAN"]),
            "DP_TO_DN":    add_basic_nic(node, NIC["CN_DP_TO_DN"]),
        }
    elif role == "DN":
        ifaces = {
            "DP": add_basic_nic(node, NIC["DN_DP"]),
        }
    else:
        raise ValueError(f"Unknown role: {role}")

    return node, ifaces


def add_l3net(slice_obj, net_name: str, interfaces):
    """Create an L3 IPv4 network."""
    return slice_obj.add_l3network(name=net_name, interfaces=interfaces, type="IPv4")

def add_l2net(slice_obj, net_name: str, interfaces):
    """Create an L2 Ethernet network."""
    return slice_obj.add_l2network(name=net_name, interfaces=interfaces)

def assign_ip(node, network_name: str, ip_pool, subnet):
    """
    Assign one IP from ip_pool to the node interface attached to network_name.

    Returns:
      (iface, addr)
    """
    iface = node.get_interface(network_name=network_name)
    addr = ip_pool.pop(0)
    iface.ip_addr_add(addr=addr, subnet=subnet)
    return iface, addr


def show_iface_and_routes(node, iface):
    """Print interface IP information and the routing table on the node."""
    node.execute(f"ip addr show {iface.get_device_name()}")
    node.execute("ip route list")

def ensure_iface_up_and_route(node, iface, subnet, ip_addr):
    # 1) bring interface up
    try:
        iface.ip_link_up()
    except Exception:
        node.ip_link_up(interface=iface)

    # 2) check current routes
    current_routes = str(node.get_ip_routes())

    subnet_str = str(subnet)

    # 3) if the connected subnet route is missing, add it explicitly
    if subnet_str not in current_routes:
        node.ip_route_add(subnet=subnet)

    print(f"[{node.get_name()}] interface {iface.get_device_name()} is set UP")
    print(f"[{node.get_name()}] ensured route for {subnet_str}")

# Wait until the node becomes reachable again
def wait_for_node(node, timeout=300, interval=10):
    start = time.time()
    while time.time() - start < timeout:
        try:
            stdout, stderr = node.execute("hostname && uptime")
            print(f"{node.get_name()} is back.")
            print(stdout)
            return True
        except Exception:
            print(f"Waiting for {node.get_name()} to come back...")
            time.sleep(interval)
    return False

### Build the Slice Topology

This cell creates the slice topology in Python. It defines the three nodes and the three networks used in the experiment.

**What this cell does:**
- Creates the `UERAN`, `CN`, and `DN` nodes
- Attaches NICs to each node
- Creates the three logical networks for:
  - **N2** control plane
  - **N3** user plane
  - **N6** data network

**Notes:**
- This step only defines the topology locally.
- It does not yet allocate resources on FABRIC.

In [ ]:
# ================
# Build slice topology (nodes, NICs, networks)
# ================

# Notes:
# - This cell only builds the slice object locally; it does NOT allocate resources yet.
# - It creates: 3 nodes, 6 NICs, and 3 L2 networks in the slice definition.

slice_obj = fablib.new_slice(name=SLICE_NAME)

# Create nodes + NIC interfaces (semantic names)
ueran_node, ueran_if = add_node(slice_obj, "UERAN")
cn_node, cn_if       = add_node(slice_obj, "CN")
dn_node, dn_if       = add_node(slice_obj, "DN")

# Create L2 networks
# add_l2net(slice_obj, NET["UERAN_CN_CP"], [ueran_if["CP"], cn_if["CP"]])
# add_l2net(slice_obj, NET["UERAN_CN_DP"], [ueran_if["DP"], cn_if["DP_TO_UERAN"]])
# add_l2net(slice_obj, NET["CN_DN"],       [cn_if["DP_TO_DN"], dn_if["DP"]])

# Create L3 networks
add_l3net(slice_obj, NET["UERAN_CN_CP"], [ueran_if["CP"], cn_if["CP"]])
add_l3net(slice_obj, NET["UERAN_CN_DP"], [ueran_if["DP"], cn_if["DP_TO_UERAN"]])
add_l3net(slice_obj, NET["CN_DN"],       [cn_if["DP_TO_DN"], dn_if["DP"]])

print("Topology object built (not submitted yet).")


### Submit the Slice

This cell submits the slice request to FABRIC and starts resource provisioning.

**What this cell does:**
- Sends the topology definition to FABRIC
- Triggers the creation of the nodes, NICs, and networks

**Notes:**
- This is the step where actual FABRIC resources are requested.
- If the selected site does not have enough free resources, slice creation may fail here.
- Provisioning may take some time depending on FABRIC site load.

In [ ]:
# ================
# Submit slice
# ================

# Notes:
# - This cell submits the slice request to FABRIC (actual resource allocation).
# - If the site is resource-constrained, this step may fail.
# - On success, the slice should appear in the FABRIC portal.

slice_obj.submit()
print("Slice submitted.")


### Retrieve the Provisioned Node Handles

After the slice has been submitted, this cell retrieves the actual provisioned node objects so the notebook can continue configuring them.

**What this cell does:**
- Gets the `UERAN`, `CN`, and `DN` node handles from the slice

**Notes:**
- This is especially useful if you rerun the notebook after the slice has already been created.

In [ ]:
# ================
# Get node handles (post-submission)
# ================

# Notes:
# - This cell ensures we are using the post-submission node objects (ready for execute/IP config).
# - If your notebook kernel restarts while the slice is still active, you can re-run from this cell.

ueran_node = slice_obj.get_node(name=NODE["UERAN"])
cn_node    = slice_obj.get_node(name=NODE["CN"])
dn_node    = slice_obj.get_node(name=NODE["DN"])

print("Node handles ready.")

### Disable Interactive `needrestart` Prompts

This cell disables interactive `needrestart` prompts on all nodes. This helps prevent package installation from stopping and waiting for manual input.

**What this cell does:**
- Updates the `needrestart` configuration on each node
- Makes later installation steps run more smoothly in notebook mode

**Notes:**
- You usually do not need to change this cell.
- This is mainly a convenience step for automation.

In [ ]:
# Disable interactive needrestart prompts on every node in the slice
cmd = r"""
set -e
sudo cp /etc/needrestart/needrestart.conf /etc/needrestart/needrestart.conf.bak
sudo sed -i \
  -e "s@#\{0,1\}\s*\$nrconf{restart} = .*@\${nrconf}{restart} = 'a';@" \
  -e "s@#\{0,1\}\s*\$nrconf{kernelhints} = .*@\${nrconf}{kernelhints} = -1;@" \
  /etc/needrestart/needrestart.conf
# grep -E "restart|kernelhints" /etc/needrestart/needrestart.conf || true
"""

for node in slice_obj.get_nodes():
    print(f"=== {node.get_name()} ===")
    node.execute(cmd)

### Clone the L25GC+ Repository

This cell clones the L25GC+ repository onto each node if it is not already present.

**What this cell does:**
- Checks whether `~/L25GC-plus` already exists
- Clones the repository from GitHub if needed

**Notes:**
- If the repository is already present, this cell will skip cloning.
- If you want to use a different fork or branch, this is the place to adjust the repository source.

In [ ]:
# Clone L25GC+ on every node in the slice
cmd = r"""
set -e
if [ ! -d ~/L25GC-plus ]; then
    cd ~ && git clone https://github.com/nycu-ucr/L25GC-plus.git
else
    echo "L25GC-plus already exists, skipping clone."
fi
"""

for node in slice_obj.get_nodes():
    print(f"=== {node.get_name()} ===")
    node.execute(cmd)

### Install MLNX_OFED on the CN Node

This cell installs MLNX_OFED on the Core Network node.

**What this cell does:**
- Runs the OFED setup script on the CN node
- Saves the installation log to `~/setup-ofed.log`

**Notes:**
- This step is only needed on the CN node.
- Installation may take some time.
- If something goes wrong, check `~/setup-ofed.log` on the CN node.

In [ ]:
# Install MLNX_OFED on the Core Network node
cmd = r"""
set -e
cd ~/L25GC-plus/scripts
yes y | bash ./install_ofed.sh > ~/setup-ofed.log 2>&1
"""

print(f"=== Installing MLNX_OFED on {cn_node.get_name()} ===")
cn_node.execute(cmd);

### Define the N2, N3, and N6 Subnets

This cell defines the IP subnets used by the three networks in the experiment.

**What this cell does:**
- Creates a subnet for **N2**
- Creates a subnet for **N3**
- Creates a subnet for **N6**

**Things you may want to change:**
- The subnet ranges, if you want to use a different addressing plan

**Notes:**
- Try to keep the three subnets non-overlapping.
- These subnet values are used later when assigning interface IP addresses.

In [ ]:
# Configure N2/N3/N6 Subnets

from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network

subnet_1_for_n2 = IPv4Network("192.168.1.0/24")
subnet_1_for_n2_available_ips = list(subnet_1_for_n2)[1:]

subnet_2_for_n3 = IPv4Network("192.168.2.0/24")
subnet_2_for_n3_available_ips = list(subnet_2_for_n3)[1:]

subnet_3_for_n6 = IPv4Network("192.168.3.0/24")
subnet_3_for_n6_available_ips = list(subnet_3_for_n6)[1:]

### Assign IP Addresses to All Interfaces

This cell assigns IP addresses to the interfaces on all nodes and brings the interfaces up.

**What this cell does:**
- Assigns IP addresses to each N2, N3, and N6 interface
- Brings interfaces up
- Ensures the expected connected routes are present
- Prints interface and routing information for verification

**Notes:**
- This is one of the key network setup steps in the notebook.
- The IPs assigned here are reused later in AMF, SMF, UPF-U, UERANSIM, and OAI configuration.
- If this step fails, many later steps will not work correctly.

In [ ]:
# ================
# Assign IPs to node interfaces
# ================

# Notes:
# - Assign IPs to all node interfaces across the three L3 networks.
# - After assigning, run `ip addr` and `ip route` to verify configuration.

# UERAN: N2
ueran_n2_iface, ueran_n2_ip = assign_ip(
    ueran_node, NET["UERAN_CN_CP"], subnet_1_for_n2_available_ips, subnet_1_for_n2
)
ensure_iface_up_and_route(
    ueran_node, ueran_n2_iface, subnet_1_for_n2, ueran_n2_ip
)
# UERAN: N3
ueran_n3_iface, ueran_n3_ip = assign_ip(
    ueran_node, NET["UERAN_CN_DP"], subnet_2_for_n3_available_ips, subnet_2_for_n3
)
ensure_iface_up_and_route(
    ueran_node, ueran_n3_iface, subnet_2_for_n3, ueran_n3_ip
)

# CN: N2
cn_n2_iface, cn_n2_ip = assign_ip(
    cn_node, NET["UERAN_CN_CP"], subnet_1_for_n2_available_ips, subnet_1_for_n2
)
ensure_iface_up_and_route(
    cn_node, cn_n2_iface, subnet_1_for_n2, cn_n2_ip
)
# CN: N3
cn_n3_iface, cn_n3_ip = assign_ip(
    cn_node, NET["UERAN_CN_DP"], subnet_2_for_n3_available_ips, subnet_2_for_n3
)
ensure_iface_up_and_route(
    cn_node, cn_n3_iface, subnet_2_for_n3, cn_n3_ip
)
# CN: N6
cn_n6_iface, cn_n6_ip = assign_ip(
    cn_node, NET["CN_DN"], subnet_3_for_n6_available_ips, subnet_3_for_n6
)
ensure_iface_up_and_route(
    cn_node, cn_n6_iface, subnet_3_for_n6, cn_n6_ip
)

# DN: N6
dn_n6_iface, dn_n6_ip = assign_ip(
    dn_node, NET["CN_DN"], subnet_3_for_n6_available_ips, subnet_3_for_n6
)
ensure_iface_up_and_route(
    dn_node, dn_n6_iface, subnet_3_for_n6, dn_n6_ip
)

show_iface_and_routes(ueran_node, ueran_n2_iface);
show_iface_and_routes(ueran_node, ueran_n3_iface);
show_iface_and_routes(cn_node, cn_n2_iface);
show_iface_and_routes(cn_node, cn_n3_iface);
show_iface_and_routes(cn_node, cn_n6_iface);
show_iface_and_routes(dn_node, dn_n6_iface);

print("IP assignment done.")
print({
    "UERAN_N2": str(ueran_n2_ip),
    "UERAN_N3": str(ueran_n3_ip),
    "CN_N2": str(cn_n2_ip),
    "CN_N3": str(cn_n3_ip),
    "CN_N6": str(cn_n6_ip),
    "DN_N6": str(dn_n6_ip),
})


### Check Basic Connectivity

This cell runs a few simple ping tests to confirm that the directly connected networks are working.

**What this cell does:**
- Tests connectivity between `UERAN` and `CN` on **N2**
- Tests connectivity between `UERAN` and `CN` on **N3**
- Tests connectivity between `CN` and `DN` on **N6**

**Notes:**
- This is a sanity check, but it is a useful one.
- If connectivity fails here, check the interface and routing setup before moving on.

In [ ]:
# ================
# (Optional) Connectivity sanity checks
# ================

# Notes:
# - This is a lightweight connectivity check using ping.
# - ICMP might be filtered in some environments; if ping fails, try TCP-based checks (iperf3/nc/curl).

# UERAN <-> CN N2 network
ueran_node.execute(f"ping -c 3 {cn_n2_ip}");
cn_node.execute(f"ping -c 3 {ueran_n2_ip}");

# UERAN <-> CN N3 network
ueran_node.execute(f"ping -c 3 {cn_n3_ip}");
cn_node.execute(f"ping -c 3 {ueran_n3_ip}");

# CN <-> DN N6 network
cn_node.execute(f"ping -c 3 {dn_n6_ip}");
dn_node.execute(f"ping -c 3 {cn_n6_ip}");

### Install L25GC+ on the CN Node

This cell installs the L25GC+ software stack on the Core Network node.

**What this cell does:**
- Runs `./scripts/setup.sh cn` on the CN node
- Saves the output to `~/setup-cn.log`

**Notes:**
- This step may take some time.
- If the installation fails, inspect `~/setup-cn.log` on the CN node.

In [ ]:
# Run the setup script to install L25GC+ on the Core Network node

cmd = r"""
# set -e
cd ~/L25GC-plus
yes y | ./scripts/setup.sh cn 
# > ~/setup-cn.log 2>&1
"""

print(f"=== Running setup on {cn_node.get_name()} ===")
cn_node.execute(cmd);

### Identify CN NICs and DPDK Port Mapping

This cell determines how the CN node’s interfaces map to PCIe devices and DPDK port indices.

**What this cell does:**
- Reads `dpdk-devbind.py -s` output
- Maps Linux interface names to PCIe devices
- Determines the DPDK port numbers used for the CN dataplane
- Prints a summary table for reference

**Notes:**
- This step is important because UPF-U needs the correct DPDK port indices for N3 and N6.
- It is better to discover these values programmatically rather than setting them manually.

In [ ]:
# Summarize Interface, PCIe, and DPDK Port Information

import pandas as pd
import re

def get_dev_pci_and_order(node):
    """
    Return:
      - dev_pci_map: dict {device_name -> PCIe address}
      - ordered_devices: list of device names in the same order as shown by dpdk-devbind.py -s
    """
    cmd = r"sudo ~/L25GC-plus/NFs/onvm-upf/subprojects/dpdk/usertools/dpdk-devbind.py -s"
    stdout, stderr = node.execute(cmd)

    if stderr:
        print(f"[{node.get_name()}] stderr from dpdk-devbind.py -s:")
        print(stderr)

    dev_pci_map = {}
    ordered_devices = []

    for line in stdout.splitlines():
        # Example:
        # 0000:07:00.0 'MT28908 Family ...' if=enp7s0 drv=mlx5_core ...
        m = re.search(r'^([0-9a-fA-F:.]+)\s+.*\bif=([^\s]+)\b', line)
        if m:
            pci_addr = m.group(1)
            dev_name = m.group(2)
            dev_pci_map[dev_name] = pci_addr
            ordered_devices.append(dev_name)

    return dev_pci_map, ordered_devices


def get_cn_dpdk_port_indices(cn_n3_dev, cn_n6_dev, ordered_devices):
    """
    Determine DPDK port index for CN N3 and N6 based on their order
    in dpdk-devbind.py -s output.

    The earlier one gets index 0, the later one gets index 1.
    """
    if cn_n3_dev not in ordered_devices:
        raise RuntimeError(f"CN N3 device {cn_n3_dev} not found in dpdk-devbind.py output")
    if cn_n6_dev not in ordered_devices:
        raise RuntimeError(f"CN N6 device {cn_n6_dev} not found in dpdk-devbind.py output")

    n3_pos = ordered_devices.index(cn_n3_dev)
    n6_pos = ordered_devices.index(cn_n6_dev)

    if n3_pos == n6_pos:
        raise RuntimeError("CN N3 and N6 devices resolved to the same device, which is unexpected")

    if n3_pos < n6_pos:
        return 0, 1
    else:
        return 1, 0


# Collect device -> PCIe mappings and device order from the CN node
cn_dev_pci, cn_ordered_devices = get_dev_pci_and_order(cn_node)

cn_n3_dev = cn_n3_iface.get_device_name()
cn_n6_dev = cn_n6_iface.get_device_name()

cn_n3_dpdk_port_index, cn_n6_dpdk_port_index = get_cn_dpdk_port_indices(
    cn_n3_dev, cn_n6_dev, cn_ordered_devices
)

print("CN device order from dpdk-devbind.py -s:")
print(cn_ordered_devices)
print(f"CN N3 device {cn_n3_dev} -> DPDK port index {cn_n3_dpdk_port_index}")
print(f"CN N6 device {cn_n6_dev} -> DPDK port index {cn_n6_dpdk_port_index}")

# ================
# Print a summary table
# ================
rows = [
    {
        "Node": ueran_node.get_name(),
        "Interface Role": "N2",
        "Network": NET["UERAN_CN_CP"],
        "Device": ueran_n2_iface.get_device_name(),
        "PCIe Address": "N/A",
        "DPDK port index": "N/A",
        "IP Address": str(ueran_n2_ip),
    },
    {
        "Node": ueran_node.get_name(),
        "Interface Role": "N3",
        "Network": NET["UERAN_CN_DP"],
        "Device": ueran_n3_iface.get_device_name(),
        "PCIe Address": "N/A",
        "DPDK port index": "N/A",
        "IP Address": str(ueran_n3_ip),
    },
    {
        "Node": cn_node.get_name(),
        "Interface Role": "N2",
        "Network": NET["UERAN_CN_CP"],
        "Device": cn_n2_iface.get_device_name(),
        "PCIe Address": cn_dev_pci.get(cn_n2_iface.get_device_name(), "N/A"),
        "DPDK port index": "N/A",
        "IP Address": str(cn_n2_ip),
    },
    {
        "Node": cn_node.get_name(),
        "Interface Role": "N3",
        "Network": NET["UERAN_CN_DP"],
        "Device": cn_n3_dev,
        "PCIe Address": cn_dev_pci.get(cn_n3_dev, "N/A"),
        "DPDK port index": cn_n3_dpdk_port_index,
        "IP Address": str(cn_n3_ip),
    },
    {
        "Node": cn_node.get_name(),
        "Interface Role": "N6",
        "Network": NET["CN_DN"],
        "Device": cn_n6_dev,
        "PCIe Address": cn_dev_pci.get(cn_n6_dev, "N/A"),
        "DPDK port index": cn_n6_dpdk_port_index,
        "IP Address": str(cn_n6_ip),
    },
    {
        "Node": dn_node.get_name(),
        "Interface Role": "N6",
        "Network": NET["CN_DN"],
        "Device": dn_n6_iface.get_device_name(),
        "PCIe Address": "N/A",
        "DPDK port index": "N/A",
        "IP Address": str(dn_n6_ip),
    },
]

iface_summary_df = pd.DataFrame(rows)
display(iface_summary_df)

### Configure the AMF

This cell updates the AMF configuration so that it listens on the correct N2 IP address.

**What this cell does:**
- Backs up the AMF config file
- Replaces the `ngapIpList` value with the CN node’s actual N2 IP
- Prints the updated configuration section

**Notes:**
- This step ensures that the gNB can reach AMF over the N2 network.
- The IP used here comes from the interface assignment step earlier in the notebook.

In [ ]:
# Configure AMF on the CN node

amf_n2_ip = str(cn_n2_ip)

cmd = f"""
set -e
cd ~/L25GC-plus
cp config/amfcfg.yaml config/amfcfg.yaml.bak

python3 - <<'PY'
from pathlib import Path
import re

cfg = Path.home() / "L25GC-plus" / "config" / "amfcfg.yaml"
text = cfg.read_text()

new_ip = "{amf_n2_ip}"

pattern = r'(ngapIpList:\\s*#.*?\\n\\s*-\\s*)([^\\n]+)'
replacement = r'\\g<1>' + new_ip

new_text, n = re.subn(pattern, replacement, text, count=1, flags=re.DOTALL)
if n == 0:
    raise RuntimeError("Failed to find ngapIpList in amfcfg.yaml")

cfg.write_text(new_text)
print(f"Updated {{cfg}}")
print(f"Set ngapIpList to: {{new_ip}}")
PY

echo "=== Updated ngapIpList section ==="
grep -A 2 "ngapIpList" ~/L25GC-plus/config/amfcfg.yaml
"""

print(f"=== Configuring AMF on {cn_node.get_name()} ===")
print(f"Setting ngapIpList to: {amf_n2_ip}")

cn_node.execute(cmd);

### Configure the SMF

This cell updates the SMF configuration so that the N3 interface matches the CN node’s actual N3 IP address.

**What this cell does:**
- Backs up the SMF config file
- Updates the N3 endpoint in `smfcfg.yaml`
- Prints the updated section for verification

**Notes:**
- This step is needed so that the user-plane path is configured correctly.
- If the N3 IP is wrong, session setup will not work properly.

In [ ]:
# Configure SMF (set the N3 endpoint to the CN node's N3 IP)

smf_n3_ip = str(cn_n3_ip)

cmd = f"""
set -e
cd ~/L25GC-plus
cp config/smfcfg.yaml config/smfcfg.yaml.bak

python3 - <<'PY'
from pathlib import Path
import re

cfg = Path.home() / "L25GC-plus" / "config" / "smfcfg.yaml"
text = cfg.read_text()

new_ip = "{smf_n3_ip}"

pattern = (
    r'(interfaceType:\\s*N3\\s*#.*?endpoints:\\s*#.*?\\n\\s*-\\s*)([^\\n]+)'
)
replacement = r'\\g<1>' + new_ip

new_text, n = re.subn(pattern, replacement, text, count=1, flags=re.DOTALL)
if n == 0:
    raise RuntimeError("Failed to find the N3 endpoints field in smfcfg.yaml")

cfg.write_text(new_text)
print(f"Updated {{cfg}}")
print(f"Set SMF N3 endpoint to: {{new_ip}}")
PY

echo "=== Updated N3 interface section ==="
grep -A 6 "interfaceType: N3" ~/L25GC-plus/config/smfcfg.yaml
"""

print(f"=== Configuring SMF on {cn_node.get_name()} ===")
print(f"CN N3 IP: {smf_n3_ip}")

cn_node.execute(cmd);

### Configure UPF-U

This cell updates the UPF-U dataplane configuration on the CN node.

**What this cell does:**
- Backs up the UPF-U config file
- Sets:
  - the UPF-U N3 IP
  - the UPF-U N6 IP
  - the gNB-side N3 IP
  - the DN-side N6 IP
  - the DPDK port indices for N3 and N6
- Prints the updated configuration

**Notes:**
- This is a key step for the user-plane datapath.
- The DPDK port numbers used here come from the NIC/PCIe discovery step earlier.
- If the port mapping is wrong, traffic may go out through the wrong interface.

In [ ]:
# Configure UPF-U on the CN node

upf_n3_ip = str(cn_n3_ip)
upf_n6_ip = str(cn_n6_ip)
an_peer_n3_ip = str(ueran_n3_ip)
dn_peer_n6_ip = str(dn_n6_ip)

n3_port_index = int(cn_n3_dpdk_port_index)
n6_port_index = int(cn_n6_dpdk_port_index)

cmd = f"""
set -e
cd ~/L25GC-plus
cp NFs/onvm-upf/5gc/upf_u/config/upf_u.yaml NFs/onvm-upf/5gc/upf_u/config/upf_u.yaml.bak

python3 - <<'PY'
from pathlib import Path
import re

cfg = Path.home() / "L25GC-plus" / "NFs" / "onvm-upf" / "5gc" / "upf_u" / "config" / "upf_u.yaml"
text = cfg.read_text()

replacements = [
    (r'(^\\s*upf_n3_ip:\\s*")[^"]*(".*$)',     r'\\g<1>{upf_n3_ip}\\g<2>'),
    (r'(^\\s*upf_n6_ip:\\s*")[^"]*(".*$)',     r'\\g<1>{upf_n6_ip}\\g<2>'),
    (r'(^\\s*an_peer_n3_ip:\\s*")[^"]*(".*$)', r'\\g<1>{an_peer_n3_ip}\\g<2>'),
    (r'(^\\s*dn_peer_n6_ip:\\s*")[^"]*(".*$)', r'\\g<1>{dn_peer_n6_ip}\\g<2>'),
    (r'(^\\s*n3_port:\\s*)\\d+(.*$)',          r'\\g<1>{n3_port_index}\\g<2>'),
    (r'(^\\s*n6_port:\\s*)\\d+(.*$)',          r'\\g<1>{n6_port_index}\\g<2>'),
]

for pattern, repl in replacements:
    new_text, n = re.subn(pattern, repl, text, count=1, flags=re.MULTILINE)
    if n == 0:
        raise RuntimeError("Failed to update pattern: " + pattern)
    text = new_text

cfg.write_text(text)

print("Updated " + str(cfg))
print("Configured values:")
print("  upf_n3_ip     = {upf_n3_ip}")
print("  upf_n6_ip     = {upf_n6_ip}")
print("  an_peer_n3_ip = {an_peer_n3_ip}")
print("  dn_peer_n6_ip = {dn_peer_n6_ip}")
print("  n3_port       = {n3_port_index}")
print("  n6_port       = {n6_port_index}")
PY

echo "=== Updated UPF-U dataplane section ==="
grep -A 12 "dataplane:" ~/L25GC-plus/NFs/onvm-upf/5gc/upf_u/config/upf_u.yaml
"""

cn_node.execute(cmd);

### Install UERANSIM and OAI UE/RAN on the UERAN Node

This cell installs the UE/RAN-side software on the UERAN node.

**What this cell does:**
- Runs `./scripts/setup.sh ue` on the UERAN node
- Saves the output to `~/setup-ueran.log`

**Notes:**
- This step prepares the node for both UERANSIM-based testing and OAI-based testing.
- If installation fails, check `~/setup-ueran.log`.

In [ ]:
# Run the setup script to install UERANsim and OAI UE/RAN on the UERAN node

cmd = r"""
set -e
cd ~/L25GC-plus
yes y | ./scripts/setup.sh ue > ~/setup-ueran.log 2>&1
"""

print(f"=== Running setup on {ueran_node.get_name()} ===")
ueran_node.execute(cmd);

### Add a Route from UERAN to the DN Server

This cell adds a route on the UERAN node so that traffic destined for the DN server goes through the CN node.

**What this cell does:**
- Adds a route toward the DN server IP
- Uses the CN node’s N3 interface IP as the next hop

**Notes:**
- This route is needed for end-to-end traffic between the UE/RAN side and the DN side.
- If you change the addressing plan, this route must stay consistent with those changes.

In [ ]:
# Add the L3 Route to DN server on the UERAN node

dn_server_ip = str(dn_n6_ip)
next_hop_ip = str(cn_n3_ip)
ueran_dev = ueran_n3_iface.get_device_name()

cmd = f"""
set -e
sudo ip route replace {dn_server_ip} via {next_hop_ip} dev {ueran_dev}
ip route
"""

print(f"=== Adding DN server route on {ueran_node.get_name()} ===")
print(f"Route: {dn_server_ip} via {next_hop_ip} dev {ueran_dev}")

ueran_node.execute(cmd);

### Configure UERANSIM gNB

This cell updates the UERANSIM gNB configuration using the actual FABRIC IP addresses assigned earlier.

**What this cell does:**
- Backs up `free5gc-gnb.yaml`
- Updates:
  - `ngapIp`
  - `gtpIp`
  - `amfConfigs.address`
- Prints the updated configuration

**Notes:**
- `ngapIp` should match the UERAN node’s N2 IP
- `gtpIp` should match the UERAN node’s N3 IP
- `amfConfigs.address` should match the CN node’s N2 IP
- These values must be correct for UERANSIM to connect to the core network successfully.

In [ ]:
# Configure UERANsim gNB (PDU Establishment only) on the UERAN node

gnb_n2_ip = str(ueran_n2_ip)
gnb_n3_ip = str(ueran_n3_ip)
amf_n2_ip = str(cn_n2_ip)

cmd = f"""
set -e

cd ~/L25GC-plus
cp UERANSIM/config/free5gc-gnb.yaml UERANSIM/config/free5gc-gnb.yaml.bak

python3 - <<'PY'
from pathlib import Path

cfg = Path.home() / "L25GC-plus" / "UERANSIM" / "config" / "free5gc-gnb.yaml"
lines = cfg.read_text().splitlines()

new_lines = []
found_ngap = False
found_gtp = False
found_amf_addr = False

in_amf_configs = False

for line in lines:
    stripped = line.lstrip()
    indent = line[:len(line) - len(stripped)]

    if stripped.startswith("ngapIp:"):
        comment = ""
        if "#" in stripped:
            comment = "  #" + stripped.split("#", 1)[1]
        line = indent + "ngapIp: {gnb_n2_ip}" + comment
        found_ngap = True

    elif stripped.startswith("gtpIp:"):
        comment = ""
        if "#" in stripped:
            comment = "  #" + stripped.split("#", 1)[1]
        line = indent + "gtpIp: {gnb_n3_ip}" + comment
        found_gtp = True

    elif stripped.startswith("amfConfigs:"):
        in_amf_configs = True

    elif in_amf_configs and stripped.startswith("- address:"):
        comment = ""
        if "#" in stripped:
            comment = "  #" + stripped.split("#", 1)[1]
        line = indent + "- address: {amf_n2_ip}" + comment
        found_amf_addr = True
        in_amf_configs = False

    new_lines.append(line)

missing = []
if not found_ngap:
    missing.append("ngapIp")
if not found_gtp:
    missing.append("gtpIp")
if not found_amf_addr:
    missing.append("amfConfigs.address")

if missing:
    raise RuntimeError("Missing keys in free5gc-gnb.yaml: " + str(missing))

cfg.write_text("\\n".join(new_lines) + "\\n")

print(f"Updated {{cfg}}")
print("  ngapIp             = {gnb_n2_ip}")
print("  gtpIp              = {gnb_n3_ip}")
print("  amfConfigs.address = {amf_n2_ip}")
PY

echo "=== Updated UERANSIM gNB config ==="
grep -A 8 "ngapIp:" ~/L25GC-plus/UERANSIM/config/free5gc-gnb.yaml
"""

print(f"=== Configuring UERANSIM gNB on {ueran_node.get_name()} ===")
print(f"ngapIp             = {gnb_n2_ip}")
print(f"gtpIp              = {gnb_n3_ip}")
print(f"amfConfigs.address = {amf_n2_ip}")

ueran_node.execute(cmd);

### Configure OAI UE/RAN for the Handover Experiment

This cell prepares the OAI-based UE/RAN setup used in the handover scenario.

**What this cell does:**
- Allocates additional N2 and N3 IP addresses for the second gNB
- Adds those IPs to the UERAN node interfaces
- Updates multiple OAI configuration files
- Sets the AMF IP, gNB IPs, and UE subscription-related parameters

**Notes:**
- This cell is mainly used for the OAI handover experiment.
- It is more complex than the UERANSIM setup because it modifies multiple files.
- The UE credentials configured here must be consistent with the subscription data used in the core network.

In [ ]:
# Configure OAI UE/RAN simulator (PDU Establishment + Handover) on the UERAN node

amf_n2_ip = str(cn_n2_ip)
gnb1_n2_ip = str(ueran_n2_ip)
gnb1_n3_ip = str(ueran_n3_ip)

# Allocate one more IP for the 2nd gNB on N2 and N3
gnb2_n2_ip = str(subnet_1_for_n2_available_ips.pop(0))
gnb2_n3_ip = str(subnet_2_for_n3_available_ips.pop(0))

ueran_n2_dev = ueran_n2_iface.get_device_name()
ueran_n3_dev = ueran_n3_iface.get_device_name()

n2_prefixlen = subnet_1_for_n2.prefixlen
n3_prefixlen = subnet_2_for_n3.prefixlen

cmd_template = r"""
set -e
cd ~/L25GC-plus

CONF_DIR=~/L25GC-plus/openairinterface5g/targets/PROJECTS/GENERIC-NR-5GC/CONF
UE_CONF=~/L25GC-plus/openairinterface5g/ci-scripts/conf_files/nrue.uicc.conf

cp "$CONF_DIR/gnb.sa.band78.fr1.106PRB.pci0.rfsim.conf" "$CONF_DIR/gnb.sa.band78.fr1.106PRB.pci0.rfsim.conf.bak"
cp "$CONF_DIR/gnb.sa.band78.fr1.106PRB.pci1.rfsim.conf" "$CONF_DIR/gnb.sa.band78.fr1.106PRB.pci1.rfsim.conf.bak"
cp "$CONF_DIR/neighbour-config-rfsim.conf" "$CONF_DIR/neighbour-config-rfsim.conf.bak"
cp "$UE_CONF" "$UE_CONF.bak"

sudo ip addr add __GNB2_N2_IP__/__N2_PREFIXLEN__ dev __UERAN_N2_DEV__ || true
sudo ip addr add __GNB2_N3_IP__/__N3_PREFIXLEN__ dev __UERAN_N3_DEV__ || true

python3 - <<'PY'
from pathlib import Path
import re

conf_dir = Path.home() / "L25GC-plus" / "openairinterface5g" / "targets" / "PROJECTS" / "GENERIC-NR-5GC" / "CONF"
ue_conf = Path.home() / "L25GC-plus" / "openairinterface5g" / "ci-scripts" / "conf_files" / "nrue.uicc.conf"

amf_n2_ip = "__AMF_N2_IP__"
gnb1_n2_ip = "__GNB1_N2_IP__"
gnb1_n3_ip = "__GNB1_N3_IP__"
gnb2_n2_ip = "__GNB2_N2_IP__"
gnb2_n3_ip = "__GNB2_N3_IP__"

plmn_line = '    plmn_list = ({ mcc = 208; mnc = 93; mnc_length = 2; snssaiList = ({ sst = 0x01; sd = 0x010203; }); });'

def update_gnb_conf(path, ng_amf_ip, ngu_ip, amf_ip):
    text = path.read_text()

    text, n1 = re.subn(
        r'^\s*plmn_list\s*=.*$',
        plmn_line,
        text,
        count=1,
        flags=re.MULTILINE
    )

    text, n2 = re.subn(
        r'(amf_ip_address\s*=\s*\(\{\s*ipv4\s*=\s*")[^"]*(".*?\}\);)',
        r'\g<1>' + amf_ip + r'\g<2>',
        text,
        count=1,
        flags=re.DOTALL
    )

    text, n3 = re.subn(
        r'(GNB_IPV4_ADDRESS_FOR_NG_AMF\s*=\s*")[^"]*(")',
        r'\g<1>' + ng_amf_ip + r'\g<2>',
        text,
        count=1
    )

    text, n4 = re.subn(
        r'(GNB_IPV4_ADDRESS_FOR_NGU\s*=\s*")[^"]*(")',
        r'\g<1>' + ngu_ip + r'\g<2>',
        text,
        count=1
    )

    if min(n1, n2, n3, n4) == 0:
        raise RuntimeError("Failed to update " + str(path))

    path.write_text(text)

update_gnb_conf(
    conf_dir / "gnb.sa.band78.fr1.106PRB.pci0.rfsim.conf",
    gnb1_n2_ip,
    gnb1_n3_ip,
    amf_n2_ip
)

update_gnb_conf(
    conf_dir / "gnb.sa.band78.fr1.106PRB.pci1.rfsim.conf",
    gnb2_n2_ip,
    gnb2_n3_ip,
    amf_n2_ip
)

nbr = conf_dir / "neighbour-config-rfsim.conf"
text = nbr.read_text()
text, n = re.subn(
    r'plmn\s*=\s*\{\s*mcc\s*=\s*\d+;\s*mnc\s*=\s*\d+;\s*mnc_length\s*=\s*\d+\s*\};',
    'plmn                = { mcc = 208; mnc = 93; mnc_length = 2 };',
    text
)
if n < 2:
    raise RuntimeError("Failed to update both PLMN entries in neighbour-config-rfsim.conf")
nbr.write_text(text)

ue_text = ue_conf.read_text()
ue_text, n1 = re.subn(
    r'imsi\s*=\s*"[^"]*";',
    'imsi = "208930000000001";',
    ue_text,
    count=1
)
ue_text, n2 = re.subn(
    r'key\s*=\s*"[^"]*";',
    'key = "8baf473f2f8fd09487cccbd7097c6862";',
    ue_text,
    count=1
)
ue_text, n3 = re.subn(
    r'opc\s*=\s*"[^"]*";',
    'opc= "8e27b6af0e692e750f32667a3b14605d";',
    ue_text,
    count=1
)
ue_text, n4 = re.subn(
    r'pdu_sessions\s*=\s*\(\{.*?\}\);',
    'pdu_sessions = ({ dnn = "internet"; nssai_sst = 0x01; nssai_sd = 0x010203; });',
    ue_text,
    count=1,
    flags=re.DOTALL
)
if min(n1, n2, n3, n4) == 0:
    raise RuntimeError("Failed to update nrue.uicc.conf")
ue_conf.write_text(ue_text)

print("Updated OAI configuration files.")
print("gNB1 N2 IP = " + gnb1_n2_ip)
print("gNB1 N3 IP = " + gnb1_n3_ip)
print("gNB2 N2 IP = " + gnb2_n2_ip)
print("gNB2 N3 IP = " + gnb2_n3_ip)
print("AMF N2 IP  = " + amf_n2_ip)
PY

echo "=== gNB1 key lines ==="
grep -E "plmn_list|amf_ip_address|GNB_IPV4_ADDRESS_FOR_NG_AMF|GNB_IPV4_ADDRESS_FOR_NGU" "$CONF_DIR/gnb.sa.band78.fr1.106PRB.pci0.rfsim.conf" || true
echo

echo "=== gNB2 key lines ==="
grep -E "plmn_list|amf_ip_address|GNB_IPV4_ADDRESS_FOR_NG_AMF|GNB_IPV4_ADDRESS_FOR_NGU" "$CONF_DIR/gnb.sa.band78.fr1.106PRB.pci1.rfsim.conf" || true
echo

echo "=== neighbour-config-rfsim.conf PLMN lines ==="
grep -n "plmn" "$CONF_DIR/neighbour-config-rfsim.conf" || true
echo

echo "=== nrue.uicc.conf key lines ==="
grep -E "imsi|key|opc|pdu_sessions" "$UE_CONF" || true
echo
"""

cmd = (
    cmd_template
    .replace("__AMF_N2_IP__", amf_n2_ip)
    .replace("__GNB1_N2_IP__", gnb1_n2_ip)
    .replace("__GNB1_N3_IP__", gnb1_n3_ip)
    .replace("__GNB2_N2_IP__", gnb2_n2_ip)
    .replace("__GNB2_N3_IP__", gnb2_n3_ip)
    .replace("__UERAN_N2_DEV__", ueran_n2_dev)
    .replace("__UERAN_N3_DEV__", ueran_n3_dev)
    .replace("__N2_PREFIXLEN__", str(n2_prefixlen))
    .replace("__N3_PREFIXLEN__", str(n3_prefixlen))
)

print(f"=== Configuring OAI UE/RAN on {ueran_node.get_name()} ===")
print(f"gNB1 N2 IP = {gnb1_n2_ip}")
print(f"gNB1 N3 IP = {gnb1_n3_ip}")
print(f"gNB2 N2 IP = {gnb2_n2_ip}")
print(f"gNB2 N3 IP = {gnb2_n3_ip}")
print(f"AMF N2 IP  = {amf_n2_ip}")

ueran_node.execute(cmd);

### Install `iperf3` Support on the DN Node

This cell installs the software needed on the DN node for traffic testing.

**What this cell does:**
- Runs `./scripts/setup.sh dn` on the DN node
- Saves the output to `~/setup-dn.log`

**Notes:**
- This step prepares the DN node for connectivity and throughput testing.
- If needed, check `~/setup-dn.log` for troubleshooting.

In [ ]:
# Run the setup script to install iperf3 on DN node

cmd = r"""
set -e
cd ~/L25GC-plus
yes y | ./scripts/setup.sh dn > ~/setup-dn.log 2>&1
"""

print(f"=== Running setup on {dn_node.get_name()} ===")
dn_node.execute(cmd);

### Add a Route from DN to the UE Subnet

This cell adds a route on the DN node so that traffic destined for the UE subnet is forwarded through the CN node.

**What this cell does:**
- Adds a route for the UE subnet `10.60.0.0/24`
- Uses the CN node’s N6 IP as the next hop
- Prints the routing table

**Notes:**
- This route is needed so that DN-to-UE traffic can return through the core network correctly.
- If you change the UE subnet in your setup, make sure to update this route as well.

In [ ]:
# Add the L3 Route to UE on the DN node

ue_ip = "10.60.0.0/24"
next_hop_ip = str(cn_n6_ip)
dn_dev = dn_n6_iface.get_device_name()

cmd = f"""
set -e
sudo ip route replace {ue_ip} via {next_hop_ip} dev {dn_dev}
ip route
"""

print(f"=== Adding UE route on {dn_node.get_name()} ===")
print(f"Route: {ue_ip} via {next_hop_ip} dev {dn_dev}")

dn_node.execute(cmd);

### Optional Slice Cleanup

This cell can be used to delete the FABRIC slice after the experiment is finished.

**What this cell does:**
- Deletes the slice and releases the resources, if the delete command is uncommented

**Notes:**
- The delete command is commented out by default.
- Only run this when you are completely done with the experiment.

In [ ]:
# slice_obj.delete()